#### Etapa 2: Modelagem com Redes Neurais (PyTorch) e Ensembles

***Objetivos desta etapa:***
1. Preparar os dados em Tensores e DataLoaders (Batching).
2. Construir e treinar uma Rede Neural (MLP) usando PyTorch com Early Stopping e tratamento de desbalanceamento (`pos_weight`).
3. Treinar um modelo Ensemble de Árvore (Random Forest) como Baseline robusto.
4. Avaliar ambos com $\ge 4$ métricas e registrar no MLflow.
5. Analisar o Trade-off de Custo de Negócio (Falso Positivo vs Falso Negativo).

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath(".."))
from src.utils.logger import get_logger
logger = get_logger("notebook_pytorch")

logger.info("1. Carregando e preparando dados para o PyTorch...")
df = pd.read_csv("../data/raw/telco_churn.csv")
df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', ''), errors='coerce').fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df_ml = pd.get_dummies(df, drop_first=True)

X = df_ml.drop(columns=['Churn']).values.astype(np.float32)
y = df_ml['Churn'].values.astype(np.float32)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

class TelcoDataset(Dataset):
    """Converte arrays NumPy para tensores do PyTorch."""
    def __init__(self, features, labels):
        self.X = torch.tensor(features)
        self.y = torch.tensor(labels).unsqueeze(1)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 64
train_loader = DataLoader(TelcoDataset(X_train_scaled, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TelcoDataset(X_val_scaled, y_val), batch_size=BATCH_SIZE, shuffle=False)

logger.info(f"Setup concluído. Features de entrada: {X_train_scaled.shape[1]}")